In [0]:
# PIPELINE RUN LOG
CATALOG = "airbnb_obs"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS
{CATALOG}.monitoring.pipeline_run_log (
    run_id STRING,      --unique per pipeline execution
    pipeline_name STRING, 
    table_name STRING,   -- table this step is monitoring
    start_time TIMESTAMP,
    end_time TIMESTAMP,
    duration_secs DOUBLE,   
    rows_processed BIGINT,
    status STRING,              -- SUCCESS / FAILED
    error_message STRING       -- null on success
) USING DELTA
          """)
print("pipeline_run_log ready")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, LongType
from datetime import datetime, timezone
import uuid, time

def log_run(pipeline_name, table_name, rows_processed, start_time, status = "SUCCESS", error_message=None):
    ended = datetime.now(timezone.utc)
    duration = (ended - start_time).total_seconds()
    run_id = start_time.strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
    row = [(run_id, pipeline_name, table_name, start_time, ended, float(duration), int(rows_processed), status, error_message)]
    schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("pipeline_name", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("start_time", TimestampType(), False),
        StructField("end_time", TimestampType(), False),
        StructField("duration_secs", DoubleType(), False),
        StructField("rows_processed", LongType(), False),
        StructField("status", StringType(), False),
        StructField("error_message", StringType(), True)
    ])
    spark.createDataFrame(row, schema).write.mode("append").saveAsTable(f"{CATALOG}.monitoring.pipeline_run_log")
    print(f" logged {pipeline_name}/{table_name}: {status}, {rows_processed} rows, {duration:.1f}s")
    return run_id


In [0]:
# LET USE IT

LANDING = "/Volumes/airbnb_obs/bronze/landing"
SOURCES = {"hosts": "hosts.csv", "listings": "listings.csv", "bookings": "bookings.csv"}

for table_name, fname in SOURCES.items():
    started = datetime.now(timezone.utc)
    try:
        df = (spark.read.option("header", True).option("inferSchema", False).csv(f"{LANDING}/{fname}").withColumn("_source_file", F.lit(fname)).withColumn("_ingested_at_utc", F.current_timestamp()))
        n = df.count()
        df.write.mode("overwrite").option("overwriteSchema","true") \
          .saveAsTable(f"{CATALOG}.bronze.{table_name}")
        log_run("bronze_ingestion", table_name, n, started, "SUCCESS")
    except Exception as e:
        log_run("bronze_ingestion", table_name, 0, started, "FAILED", str(e))
        raise

In [0]:
#CREATE VIEW

spark.sql(f"""
CREATE OR REPLACE VIEW {CATALOG}.monitoring.v_current_dq_state AS
WITH latest AS (
    SELECT table_name, MAX(run_id) AS latest_run
    FROM {CATALOG}.monitoring.dq_results
    GROUP BY table_name
)
SELECT r.table_name, r.rule_name, r.status, r.failed_records, r.total_records, r.check_ts
FROM {CATALOG}.monitoring.dq_results r
JOIN latest l
  ON r.table_name = l.table_name AND r.run_id = l.latest_run
ORDER BY r.table_name, r.status DESC
""")

display(spark.table(f"{CATALOG}.monitoring.v_current_dq_state"))
display(spark.table(f"{CATALOG}.monitoring.pipeline_run_log"))